# Asistente de refacciones GRTR — fase 2: PLN, visión y primera integración

Jaime Eduardo Silva Navarro (T05207098) · Maestría en Inteligencia Artificial, Universidad Tecmilenio

Desarrollo de aplicaciones con inteligencia artificial · Avance de proyecto 2

La fase 1 identificaba la intención de cada mensaje con reglas. Esta fase agrega un **clasificador de intenciones**
(preprocesamiento con NLTK, bolsa de palabras o TF-IDF, Naive Bayes o SVM), un **clasificador de imágenes** de piezas
(ResNet50 con transferencia de aprendizaje) y una **primera integración**: el mismo asistente atiende texto y fotos.
El catálogo y las frases de entrenamiento son **sintéticos**; las fotos son de un conjunto público de Kaggle (licencia Apache 2.0).

## 1. Entorno

In [ ]:
import sys, os, json, re, random, hashlib
import numpy as np
import nltk
import sklearn

print("Python", sys.version.split()[0], "· NLTK", nltk.__version__, "· scikit-learn", sklearn.__version__)
assert sys.version_info >= (3, 10), "Se requiere Python 3.10 o superior"
for recurso in ("punkt_tab", "stopwords", "cess_esp"):
    nltk.download(recurso, quiet=True)
SEMILLA = 42
random.seed(SEMILLA)
np.random.seed(SEMILLA)

## 2. Datos del repositorio
En Colab se clona el repositorio; si los archivos ya están en la carpeta, se usan. El chatbot de la fase 1 se importa
de `asistente_fase1.py`, que tiene sin cambios el código de su notebook.

In [ ]:
import subprocess

REPO_URL = "https://github.com/AutomIA-TR/asistente-refacciones-grtr.git"
CARPETA = "asistente-refacciones-grtr"
if not os.path.exists("datos/base_conocimiento.json"):
    if not os.path.isdir(CARPETA):
        subprocess.run(["git", "clone", "-q", REPO_URL, CARPETA], check=True)
    os.chdir(CARPETA)
sys.path.insert(0, os.getcwd())
print("Carpeta de trabajo:", os.getcwd())

from asistente_fase1 import BaseConocimiento, Chatbot, ClasificadorReglas, sin_acentos

base, catalogo = BaseConocimiento().leer()
with open("datos/intenciones_entrenamiento.json", encoding="utf-8") as f:
    DATOS = json.load(f)
with open("pruebas/conversaciones.json", encoding="utf-8") as f:
    PRUEBAS = json.load(f)
with open("pruebas/frases_limite_fase2.json", encoding="utf-8") as f:
    LIMITE = json.load(f)["frases"]
INTENCIONES = DATOS["intenciones"]
textos = [d["texto"] for d in DATOS["frases"]]
etiquetas = [d["intencion"] for d in DATOS["frases"]]
TURNOS = [(t["usuario"], t["intencion"]) for c in PRUEBAS["funcionales"] for t in c["turnos"]]
print(f"{len(INTENCIONES)} intenciones · {len(catalogo)} productos · {len(TURNOS)} turnos de la fase 1 · {len(LIMITE)} frases de límite")

## 3. Conjunto de datos textual
Frases sintéticas de clientes, escritas para cada una de las 14 intenciones del Avance 1. Se comprueba que no haya frases
repetidas ni frases que también estén en las pruebas (los 43 turnos de la fase 1 y las frases de límite).

In [ ]:
def clave(t):
    """Forma de comparar frases sin mayúsculas, acentos ni signos."""
    return " ".join(re.sub(r"[^a-z0-9\s]", " ", sin_acentos(t.lower())).split())


conteo = {i: etiquetas.count(i) for i in INTENCIONES}
print(f"{len(textos)} frases en {len(INTENCIONES)} intenciones\n")
for i in INTENCIONES:
    print(f"  {i:<22}{conteo[i]:>4}   p. ej.: {textos[etiquetas.index(i)]}")
claves = [clave(t) for t in textos]
repetidas = len(claves) - len(set(claves))
de_pruebas = {clave(t) for t, _ in TURNOS} | {clave(f["texto"]) for f in LIMITE}
en_pruebas = sorted(set(claves) & de_pruebas)
print(f"\nFrases repetidas: {repetidas} · frases que también están en las pruebas: {len(en_pruebas)}")
assert repetidas == 0 and not en_pruebas

## 4. Partición en entrenamiento y prueba
75 % para entrenar y 25 % para probar, estratificada por intención y con semilla fija. Todo lo que se aprende de los
datos (el vocabulario de la corrección, la representación y el modelo) sale sólo de la parte de entrenamiento.

In [ ]:
from sklearn.model_selection import train_test_split

X_ent, X_pru, y_ent, y_pru = train_test_split(textos, etiquetas, test_size=0.25, random_state=SEMILLA, stratify=etiquetas)
print(f"Entrenamiento: {len(X_ent)} frases · prueba: {len(X_pru)} frases")
print("Frases de prueba por intención:", {i: y_pru.count(i) for i in INTENCIONES})

## 5. Preprocesamiento del texto
Cada mensaje pasa por las cuatro técnicas, en este orden:
1. **Normalización**: minúsculas, sin acentos ni signos; las referencias, los años y los números se cambian por una
   etiqueta (`zref`, `zanio`, `znum`), porque a la intención no le importa cuál es el código, sino que hay uno.
2. **Tokenización** con `word_tokenize` de NLTK.
3. Normalización de cada palabra: **corrección aproximada** contra el vocabulario de entrenamiento (la mejora L1 del
   Avance 1: «bomva» → «bomba») sólo para palabras que no existen en un léxico del español (CESS-ESP de NLTK), y
   etiqueta `zvehiculo` para los vehículos.
4. **Eliminación de palabras vacías** con la lista de NLTK, personalizada: se conservan las negaciones y las palabras
   de pregunta («no», «qué», «cuánto», «cómo», «dónde»), que cambian la intención. La lista se eligió con validación
   cruzada sobre el entrenamiento (sección 8).
5. **Stemming** con `SnowballStemmer` (NLTK no trae lematizador del español); los nombres de pieza del catálogo se
   cambian por la etiqueta `zpieza`, ya en raíces, para que «bombas de dirección» y «bomba de dirección» coincidan.

In [ ]:
import difflib
from nltk.corpus import cess_esp, stopwords
from nltk.stem import SnowballStemmer
from nltk.tokenize import word_tokenize

# negaciones y palabras de pregunta: están en la lista de NLTK, pero cambian la intención (se eligió con validación cruzada)
PROTEGIDAS = {"no", "ni", "sin", "nunca", "que", "cual", "cuales", "quien", "quienes", "cuanto", "cuanta", "cuantos",
              "cuantas", "como", "cuando", "donde"}
ETIQUETAS = {"zref", "zanio", "znum", "zvehiculo", "zpieza"}
# léxico del español (corpus CESS-ESP de NLTK): una palabra que existe no se corrige, aunque no esté en el entrenamiento
LEXICO = {sin_acentos(w.lower()) for w in cess_esp.words() if w.isalpha()}


class PreprocesadorPLN:
    """Normalización, tokenización, palabras vacías y stemming; cada etapa se puede apagar para medir su efecto."""

    def __init__(self, base, corregir=True, quitar_vacias=True, raices=True, etiquetas=True, protegidas=PROTEGIDAS):
        self.stemmer = SnowballStemmer("spanish")
        self.corregir_on, self.quitar_vacias, self.raices_on, self.etiquetas = corregir, quitar_vacias, raices, etiquetas
        self.vacias = {sin_acentos(p) for p in stopwords.words("spanish")} - set(protegidas)
        variantes = [sin_acentos(v) for vs in base["vehiculos"].values() for v in vs]
        self.con_guion = {v: v.replace("-", "") for v in variantes if "-" in v}
        self.vehiculos = {v.replace("-", "") for v in variantes}
        nombres = [s for lista in base["familias"].values() for s in lista] + list(base["genericos"])
        sec = [[w for w in self.tokenizar(self.normalizar(s, etiquetas=False)) if w not in self.vacias] for s in nombres]
        self.piezas_raiz = sorted({tuple(self.stemmer.stem(w) for w in s) for s in sec}, key=len, reverse=True)
        self.piezas_token = sorted({tuple(s) for s in sec}, key=len, reverse=True)
        self.base_vocab = {w for s in sec for w in s} | self.vehiculos | self.vacias | set(protegidas)
        self.vocabulario = sorted(self.base_vocab)

    def ajustar(self, frases):
        """El vocabulario de la corrección sale sólo de las frases de entrenamiento."""
        palabras = {w for f in frases for w in self.tokenizar(self.normalizar(f))}
        self.vocabulario = sorted(self.base_vocab | {w for w in palabras if w not in ETIQUETAS})
        self._conocidas = set(self.vocabulario) | LEXICO
        self._raices_conocidas = {self.stemmer.stem(w) for w in self._conocidas}
        return self

    def normalizar(self, texto, etiquetas=None):
        etiquetas = self.etiquetas if etiquetas is None else etiquetas
        t = re.sub(r"[^a-z0-9\s-]", " ", sin_acentos(texto.lower()))
        for v, sin_guion in self.con_guion.items():
            t = re.sub(r"\b" + re.escape(v) + r"\b", sin_guion, t)
        if etiquetas:
            t = re.sub(r"\b[a-z]{2}-?\d{4}\b", " zref ", t)
            t = re.sub(r"\b(19[5-9]\d|20[0-4]\d)\b", " zanio ", t)
            t = re.sub(r"\b\d+\b", " znum ", t)
        return re.sub(r"[\s-]+", " ", t).strip()

    @staticmethod
    def tokenizar(texto):
        return [t for t in word_tokenize(texto, language="spanish") if re.search(r"[a-z0-9]", t)]

    def corregir(self, palabra):
        if not self.corregir_on or palabra in ETIQUETAS or len(palabra) < 4 or not palabra.isalpha():
            return palabra
        if palabra in self._conocidas or self.stemmer.stem(palabra) in self._raices_conocidas:
            return palabra
        parecidas = difflib.get_close_matches(palabra, self.vocabulario, n=1, cutoff=0.8)
        return parecidas[0] if parecidas else palabra

    def corregir_texto(self, texto):
        """El mensaje corregido, sin etiquetas, para que el extractor de entidades de la fase 1 lo lea."""
        limpio = re.sub(r"[^a-z0-9\s-]", " ", sin_acentos(texto.lower()))   # conserva el guion de las referencias
        return " ".join(self.corregir(p) for p in limpio.split())

    @staticmethod
    def etiquetar_piezas(seq, piezas):
        salida, i = [], 0
        while i < len(seq):
            for p in piezas:
                if p and tuple(seq[i:i + len(p)]) == p:
                    salida.append("zpieza")
                    i += len(p)
                    break
            else:
                salida.append(seq[i])
                i += 1
        return salida

    def etapas(self, texto):
        norm = self.normalizar(texto)
        tokens = self.tokenizar(norm)
        corregidos = [self.corregir(t) for t in tokens]
        if self.etiquetas:
            corregidos = ["zvehiculo" if t in self.vehiculos else t for t in corregidos]
        sin_vacias = [t for t in corregidos if not (self.quitar_vacias and t in self.vacias)]
        if self.raices_on:
            finales = [t if t in ETIQUETAS else self.stemmer.stem(t) for t in sin_vacias]
            piezas = self.piezas_raiz
        else:
            finales, piezas = list(sin_vacias), self.piezas_token
        if self.etiquetas:
            finales = self.etiquetar_piezas(finales, piezas)
        return {"normalizado": norm, "tokens": tokens, "corregidos": corregidos, "sin_vacias": sin_vacias, "final": finales}

    def procesar(self, texto):
        return " ".join(self.etapas(texto)["final"])


pre = PreprocesadorPLN(base).ajustar(X_ent)
print(f"Vocabulario de la corrección: {len(pre.vocabulario)} palabras (de las frases de entrenamiento y la base)")
EJEMPLOS = ["bomva de gazolina para tzuru", "¿Tienen la BE-1126 para un Jetta 2005?",
            "¿No aceptan depósito bancario?", "Busco bombas de dirección para la lobo 2012"]
for e in EJEMPLOS:
    et = pre.etapas(e)
    print(f"\nOriginal:      {e}")
    print(f"1 Normalizado: {et['normalizado']}")
    print(f"2 Tokens:      {et['tokens']}")
    print(f"3 Corregidos:  {et['corregidos']}")
    print(f"4 Sin vacías:  {et['sin_vacias']}")
    print(f"5 Raíces:      {et['final']}")
P_ent = [pre.procesar(t) for t in X_ent]
P_pru = [pre.procesar(t) for t in X_pru]

## 6. Representación del texto: bolsa de palabras y TF-IDF
Las frases preprocesadas se convierten en vectores con `CountVectorizer` (bolsa de palabras) y `TfidfVectorizer`,
con palabras sueltas y pares de palabras. Los términos de mayor peso de cada intención muestran qué aprende el modelo.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer


def vectorizador(tipo):
    clase = TfidfVectorizer if tipo == "TF-IDF" else CountVectorizer
    return clase(ngram_range=(1, 2), token_pattern=r"\S+", lowercase=False)


tfidf = vectorizador("TF-IDF").fit(P_ent)
M = tfidf.transform(P_ent)
print(f"Matriz TF-IDF de entrenamiento: {M.shape[0]} frases × {M.shape[1]} términos · "
      f"{100 * M.nnz / (M.shape[0] * M.shape[1]):.2f} % de celdas distintas de cero")
terminos = np.array(tfidf.get_feature_names_out())
y_arr = np.array(y_ent)
print("\nTérminos con mayor peso TF-IDF promedio por intención:")
for i in INTENCIONES:
    media = np.asarray(M[y_arr == i].mean(axis=0)).ravel()
    print(f"  {i:<22}", ", ".join(terminos[np.argsort(-media)[:5]]))

## 7. Clasificador de intenciones: cuatro experimentos
Como en el Tema 6: bolsa de palabras o TF-IDF, cada una con Naive Bayes multinomial o SVM lineal. Se comparan con
validación cruzada estratificada de 5 partes sobre el entrenamiento (F1 macro) y el mejor se evalúa una sola vez en la prueba.

In [ ]:
from sklearn.base import clone
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             precision_recall_fscore_support)
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

EXPERIMENTOS = {
    "BoW + Naive Bayes": Pipeline([("vector", vectorizador("BoW")), ("modelo", MultinomialNB())]),
    "TF-IDF + Naive Bayes": Pipeline([("vector", vectorizador("TF-IDF")), ("modelo", MultinomialNB())]),
    "BoW + SVM lineal": Pipeline([("vector", vectorizador("BoW")), ("modelo", LinearSVC(random_state=SEMILLA))]),
    "TF-IDF + SVM lineal": Pipeline([("vector", vectorizador("TF-IDF")), ("modelo", LinearSVC(random_state=SEMILLA))]),
}
PARTES = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)
resultados_cv = {}
print(f"{'Experimento':<24}{'F1 macro (VC)':>15}{'Exactitud (VC)':>16}")
for nombre, pipe in EXPERIMENTOS.items():
    r = cross_validate(clone(pipe), P_ent, y_ent, cv=PARTES, scoring=["f1_macro", "accuracy"])
    resultados_cv[nombre] = (r["test_f1_macro"].mean(), r["test_accuracy"].mean())
    print(f"{nombre:<24}{resultados_cv[nombre][0]:>15.3f}{resultados_cv[nombre][1]:>16.3f}")
MEJOR = max(resultados_cv, key=lambda n: resultados_cv[n][0])
print(f"\nElegido por F1 macro en validación cruzada: {MEJOR}")
modelo_texto = clone(EXPERIMENTOS[MEJOR]).fit(P_ent, y_ent)
pred_pru = modelo_texto.predict(P_pru)
p, r, f1, _ = precision_recall_fscore_support(y_pru, pred_pru, average="macro", zero_division=0)
EXACTITUD_TEXTO = accuracy_score(y_pru, pred_pru)
print(f"\nPrueba ({len(y_pru)} frases): exactitud {EXACTITUD_TEXTO:.3f} · precisión {p:.3f} · recall {r:.3f} · F1 {f1:.3f} (macro)\n")
print(classification_report(y_pru, pred_pru, labels=INTENCIONES, digits=3, zero_division=0))

In [ ]:
import matplotlib.pyplot as plt

MC_TEXTO = confusion_matrix(y_pru, pred_pru, labels=INTENCIONES)
fig, ax = plt.subplots(figsize=(10, 9))
ConfusionMatrixDisplay(MC_TEXTO, display_labels=INTENCIONES).plot(ax=ax, cmap="Greens", colorbar=False, xticks_rotation=90)
ax.set_title(f"Matriz de confusión de intenciones · {MEJOR} · prueba")
plt.tight_layout()
plt.show()
errores = [(t, y, q) for t, y, q in zip(X_pru, y_pru, pred_pru) if y != q]
print(f"Frases de prueba mal clasificadas: {len(errores)}")
for t, y, q in errores:
    print(f"  «{t}»: {y} → {q}")

## 8. Validación del preprocesamiento
El Tema 5 pide validar cada decisión de preprocesamiento con métricas. Se repite el mejor experimento apagando una etapa a
la vez y se mide la F1 macro en validación cruzada, la exactitud en la prueba y los aciertos en las frases de límite.

In [ ]:
def acierta_limite(modelo, prepro):
    pred = modelo.predict([prepro.procesar(f["texto"]) for f in LIMITE])
    return sum(q in f["aceptables"] for q, f in zip(pred, LIMITE))


VARIANTES = {"Completo": {}, "Sin corrección aproximada": {"corregir": False},
             "Sin quitar palabras vacías": {"quitar_vacias": False},
             "Palabras vacías de NLTK sin personalizar": {"protegidas": set()}, "Sin stemming": {"raices": False},
             "Sin etiquetas de entidades": {"etiquetas": False}}
ABLACION = {}
print(f"{'Variante':<42}{'F1 macro (VC)':>14}{'Exactitud prueba':>18}{'Límite':>10}")
for nombre, opciones in VARIANTES.items():
    pr = PreprocesadorPLN(base, **opciones).ajustar(X_ent)
    pe = [pr.procesar(t) for t in X_ent]
    cv = cross_validate(clone(EXPERIMENTOS[MEJOR]), pe, y_ent, cv=PARTES, scoring="f1_macro")["test_score"].mean()
    m = clone(EXPERIMENTOS[MEJOR]).fit(pe, y_ent)
    ex = accuracy_score(y_pru, m.predict([pr.procesar(t) for t in X_pru]))
    ABLACION[nombre] = (cv, ex, acierta_limite(m, pr))
    print(f"{nombre:<42}{cv:>14.3f}{ex:>18.3f}{ABLACION[nombre][2]:>7}/{len(LIMITE)}")

## 9. Reglas de la fase 1 contra el clasificador
Las mismas frases con los dos métodos, sólo la intención: los 43 turnos de las diez conversaciones del Avance 1 (escritos
antes que las frases de entrenamiento) y las frases de límite (faltas de ortografía, lenguaje coloquial, choques de
palabras clave y piezas que no están en el catálogo).

In [ ]:
reglas = ClasificadorReglas(base, catalogo)
r_turnos = sum(reglas.identificar(t)[0] == i for t, i in TURNOS)
m_turnos = sum(q == i for q, (_, i) in zip(modelo_texto.predict([pre.procesar(t) for t, _ in TURNOS]), TURNOS))
print(f"43 turnos del Avance 1 · reglas: {r_turnos}/{len(TURNOS)} · clasificador: {m_turnos}/{len(TURNOS)}")
for (t, i), q in zip(TURNOS, modelo_texto.predict([pre.procesar(t) for t, _ in TURNOS])):
    if q != i:
        print(f"  clasificador falla «{t}»: {i} → {q}")
print(f"\n{'Id':<4}{'Tipo':<38}{'Reglas':<20}{'Clasificador':<20}Frase")
r_lim = m_lim = 0
for f in LIMITE:
    a = reglas.identificar(f["texto"])[0]
    b = modelo_texto.predict([pre.procesar(f["texto"])])[0]
    r_lim += a in f["aceptables"]
    m_lim += b in f["aceptables"]
    marca = lambda x: x + (" ✓" if x in f["aceptables"] else " ✗")
    print(f"{f['id']:<4}{f['tipo']:<38}{marca(a):<20}{marca(b):<20}{f['texto']}")
print(f"\nFrases de límite · reglas: {r_lim}/{len(LIMITE)} · clasificador: {m_lim}/{len(LIMITE)}")

## 10. Conjunto de imágenes
«50 Types of Car Parts — Image Classification» (Kaggle, `gpiosenka/car-parts-40-classes`, licencia Apache 2.0): fotos de
224 × 224 de 50 tipos de pieza. Se usan **10 clases**: cinco que corresponden a familias del catálogo y cinco que no se
venden, para que el asistente aprenda a decir «eso no lo manejamos». Las fotos no se guardan en el repositorio: se bajan
en cada corrida. Si la descarga pide credenciales, se guarda la clave de Kaggle en los Secretos de Colab
(`KAGGLE_USERNAME` y `KAGGLE_KEY`, o `KAGGLE_API_TOKEN`) con acceso del notebook activado.

In [ ]:
try:
    import kagglehub
except ImportError:   # Colab suele traerlo; si no, se instala
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"], check=True)
    import kagglehub

SLUG = "gpiosenka/car-parts-40-classes"
CLASES = {  # clase del conjunto: (nombre para el cliente, familia del catálogo o None si no se vende)
    "OXYGEN SENSOR": ("sensor de oxígeno", "SO"),
    "IGNITION COIL": ("bobina de encendido", "BE"),
    "FUEL INJECTOR": ("inyector", "IN"),
    "STARTER": ("marcha (motor de arranque)", "MA"),
    "OVERFLOW TANK": ("depósito de agua", "DA"),
    "SPARK PLUG": ("bujía", None),
    "OIL PRESSURE SENSOR": ("sensor de presión de aceite", None),
    "ALTERNATOR": ("alternador", None),
    "WATER PUMP": ("bomba de agua", None),
    "OIL FILTER": ("filtro de aceite", None),
}
try:
    RUTA_FOTOS = kagglehub.dataset_download(SLUG)
except Exception as e:
    print("No se pudo bajar el conjunto:", type(e).__name__, e)
    print("Guarda la clave de Kaggle en los Secretos de Colab y vuelve a correr esta celda.")
    raise
print("Conjunto en:", RUTA_FOTOS)


def nombre_clase(c):
    return " ".join(c.upper().split())


todas = []
for carpeta, _, archivos in os.walk(RUTA_FOTOS):
    for a in sorted(archivos):
        if a.lower().endswith((".jpg", ".jpeg", ".png")):
            todas.append((os.path.join(carpeta, a), nombre_clase(os.path.basename(carpeta))))
todas.sort()
disponibles = sorted({c for _, c in todas})
print(f"{len(todas)} fotos en {len(disponibles)} clases")
faltan = [c for c in CLASES if c not in disponibles]
if faltan:
    print("Clases disponibles:", disponibles)
    raise ValueError(f"No encontré estas clases en el conjunto: {faltan}")

In [ ]:
from collections import Counter


def huella(ruta):
    with open(ruta, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()


elegidas = [(r, c) for r, c in todas if c in CLASES]
grupos = {}
for r, c in elegidas:
    grupos.setdefault(huella(r), []).append((r, c))
unicas, repetidas_img, ambiguas = [], 0, 0
for h in sorted(grupos):
    g = grupos[h]
    if len({c for _, c in g}) > 1:   # la misma foto con dos etiquetas: se descarta completa
        ambiguas += len(g)
        continue
    unicas.append(g[0])
    repetidas_img += len(g) - 1
unicas.sort()
print(f"Fotos de las 10 clases: {len(elegidas)} · copias idénticas descartadas: {repetidas_img} · "
      f"con dos etiquetas: {ambiguas} · únicas: {len(unicas)}")
y_img = [c for _, c in unicas]
entrenamiento, resto = train_test_split(unicas, test_size=0.30, random_state=SEMILLA, stratify=y_img)
validacion, prueba = train_test_split(resto, test_size=0.50, random_state=SEMILLA, stratify=[c for _, c in resto])
PARTICIONES = {"entrenamiento": entrenamiento, "validación": validacion, "prueba": prueba}
assert not ({huella(r) for r, _ in entrenamiento} & {huella(r) for r, _ in validacion + prueba})
print(f"\n{'Clase':<22}{'Para el cliente':<30}{'Familia':<9}{'Entren.':>8}{'Valid.':>8}{'Prueba':>8}")
for c, (nombre, fam) in CLASES.items():
    n = [sum(1 for _, x in parte if x == c) for parte in PARTICIONES.values()]
    print(f"{c:<22}{nombre:<30}{fam or '—':<9}{n[0]:>8}{n[1]:>8}{n[2]:>8}")
print(f"{'Total':<61}{len(entrenamiento):>8}{len(validacion):>8}{len(prueba):>8}")

In [ ]:
from PIL import Image

fig, ejes = plt.subplots(2, 5, figsize=(14, 6))
for eje, c in zip(ejes.ravel(), CLASES):
    ruta = next(r for r, x in entrenamiento if x == c)
    eje.imshow(Image.open(ruta).convert("RGB"))
    eje.set_title(CLASES[c][0], fontsize=10)
    eje.axis("off")
plt.suptitle("Una foto de entrenamiento por clase")
plt.tight_layout()
plt.show()

## 11. Arquitectura y entrenamiento del clasificador visual
Como en el ejercicio del Tema 9: **ResNet50** preentrenada en ImageNet, sin su capa superior y congelada, con una cabeza
nueva (promedio global, *dropout* y *softmax* de 10 clases). Primero se entrena sólo la cabeza; después se descongelan las
últimas 30 capas con una tasa de aprendizaje baja (*fine-tuning* parcial). El aumento de datos (volteo, rotación, zoom y
contraste) se aplica sólo al entrenamiento (Tema 8), y el desbalance se compensa con pesos por clase (Tema 10).

In [ ]:
import tensorflow as tf
import keras
from keras import layers

keras.utils.set_random_seed(SEMILLA)
print("TensorFlow", tf.__version__, "· Keras", keras.__version__, "· GPU:", tf.config.list_physical_devices("GPU") or "no")
TAMANO = 224
NOMBRES = list(CLASES)
INDICE = {c: i for i, c in enumerate(NOMBRES)}
aumento = keras.Sequential([layers.RandomFlip("horizontal"), layers.RandomRotation(0.08), layers.RandomZoom(0.1),
                            layers.RandomContrast(0.1)], name="aumento")


def cargar(ruta):
    img = tf.io.decode_image(tf.io.read_file(ruta), channels=3, expand_animations=False)
    return tf.image.resize(img, (TAMANO, TAMANO))


def lotes(particion, entrenar=False):
    rutas = [r for r, _ in particion]
    ys = [INDICE[c] for _, c in particion]
    ds = tf.data.Dataset.from_tensor_slices((rutas, ys))
    if entrenar:
        ds = ds.shuffle(len(rutas), seed=SEMILLA)
    ds = ds.map(lambda r, y: (cargar(r), y), num_parallel_calls=tf.data.AUTOTUNE).batch(32)
    if entrenar:
        ds = ds.map(lambda x, y: (aumento(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(lambda x, y: (keras.applications.resnet50.preprocess_input(x), y))
    return ds.prefetch(tf.data.AUTOTUNE)


ds_ent, ds_val, ds_pru = lotes(entrenamiento, entrenar=True), lotes(validacion), lotes(prueba)
cuenta = Counter(INDICE[c] for _, c in entrenamiento)
PESOS_CLASE = {i: len(entrenamiento) / (len(NOMBRES) * cuenta[i]) for i in range(len(NOMBRES))}
print("Pesos por clase:", {NOMBRES[i]: round(w, 3) for i, w in PESOS_CLASE.items()})

base_cnn = keras.applications.ResNet50(weights="imagenet", include_top=False, input_shape=(TAMANO, TAMANO, 3))
base_cnn.trainable = False
entrada = keras.Input((TAMANO, TAMANO, 3))
x = base_cnn(entrada, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
salida = layers.Dense(len(NOMBRES), activation="softmax")(x)
cnn = keras.Model(entrada, salida, name="resnet50_refacciones")
cnn.compile(optimizer=keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
entrenables = sum(int(np.prod(w.shape)) for w in cnn.trainable_weights)
print(f"Parámetros: {cnn.count_params():,} en total · {entrenables:,} entrenables (sólo la cabeza)")

In [ ]:
def llamadas():
    return [keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6)]


EPOCAS_A = 15
print("Etapa A: transferencia de aprendizaje (base congelada)")
hist_a = cnn.fit(ds_ent, validation_data=ds_val, epochs=EPOCAS_A, class_weight=PESOS_CLASE, callbacks=llamadas(), verbose=2)
VAL_A = cnn.evaluate(ds_val, verbose=0)[1]
pesos_a = cnn.get_weights()
prob_a = cnn.predict(ds_pru, verbose=0)

base_cnn.trainable = True
for capa in base_cnn.layers[:-30]:
    capa.trainable = False
for capa in base_cnn.layers:
    if isinstance(capa, layers.BatchNormalization):
        capa.trainable = False
cnn.compile(optimizer=keras.optimizers.Adam(1e-5), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
EPOCAS_B = 10
print("\nEtapa B: fine-tuning de las últimas 30 capas (tasa 1e-5)")
hist_b = cnn.fit(ds_ent, validation_data=ds_val, epochs=EPOCAS_B, class_weight=PESOS_CLASE, callbacks=llamadas(), verbose=2)
VAL_B = cnn.evaluate(ds_val, verbose=0)[1]
prob_b = cnn.predict(ds_pru, verbose=0)
if VAL_B >= VAL_A:
    ETAPA_FINAL, prob_final = "B (fine-tuning)", prob_b
else:
    cnn.set_weights(pesos_a)
    ETAPA_FINAL, prob_final = "A (base congelada)", prob_a
print(f"\nExactitud en validación · A: {VAL_A:.3f} · B: {VAL_B:.3f} → se usa la etapa {ETAPA_FINAL}")

curvas = {k: hist_a.history[k] + hist_b.history[k] for k in ("accuracy", "val_accuracy", "loss", "val_loss")}
fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
for eje, m, titulo in ((ejes[0], "accuracy", "Exactitud"), (ejes[1], "loss", "Pérdida")):
    eje.plot(curvas[m], label="entrenamiento")
    eje.plot(curvas["val_" + m], label="validación")
    eje.axvline(len(hist_a.history["loss"]) - 0.5, color="gray", linestyle="--")
    eje.set_title(titulo + " por época (la línea separa A y B)")
    eje.set_xlabel("época")
    eje.legend()
plt.tight_layout()
plt.show()

## 12. Evaluación del clasificador visual
Las cinco evidencias sobre las fotos de **prueba**, que el modelo no vio: exactitud, precisión, recall, F1 por clase y
matriz de confusión. Se reporta también la exactitud de la etapa A para comparar la transferencia sola contra el *fine-tuning*.

In [ ]:
y_img_pru = np.array([INDICE[c] for _, c in prueba])
pred_img = prob_final.argmax(axis=1)
NOMBRES_ES = [CLASES[c][0] for c in NOMBRES]
EXACTITUD_A = accuracy_score(y_img_pru, prob_a.argmax(axis=1))
EXACTITUD_B = accuracy_score(y_img_pru, prob_b.argmax(axis=1))
EXACTITUD_IMG = accuracy_score(y_img_pru, pred_img)
pi, ri, fi, _ = precision_recall_fscore_support(y_img_pru, pred_img, average="macro", zero_division=0)
print(f"Prueba ({len(y_img_pru)} fotos) · exactitud A: {EXACTITUD_A:.3f} · B: {EXACTITUD_B:.3f}")
print(f"Modelo final, etapa {ETAPA_FINAL}: exactitud {EXACTITUD_IMG:.3f} · precisión {pi:.3f} · recall {ri:.3f} · F1 {fi:.3f} (macro)\n")
print(classification_report(y_img_pru, pred_img, labels=list(range(len(NOMBRES))), target_names=NOMBRES_ES,
                            digits=3, zero_division=0))
MC_IMG = confusion_matrix(y_img_pru, pred_img, labels=list(range(len(NOMBRES))))
fig, ax = plt.subplots(figsize=(9, 8))
ConfusionMatrixDisplay(MC_IMG, display_labels=NOMBRES_ES).plot(ax=ax, cmap="Greens", colorbar=False, xticks_rotation=60)
ax.set_title("Matriz de confusión de piezas · prueba")
plt.tight_layout()
plt.show()
mal = [(r, NOMBRES[y], NOMBRES[q], prob_final[k, q]) for k, ((r, _), y, q) in enumerate(zip(prueba, y_img_pru, pred_img)) if y != q]
print(f"Fotos de prueba mal clasificadas: {len(mal)}")
for r, y, q, pq in mal:
    print(f"  {os.path.basename(r)}: {CLASES[y][0]} → {CLASES[q][0]} ({pq:.0%})")

## 13. Integración texto-imagen
El asistente de la fase 1 (sus entidades, su estado y sus respuestas) recibe ahora dos entradas:
- **Texto**: el clasificador elige la intención; si su confianza es baja, el asistente no adivina y responde con la salida de
  fuera de alcance. Las reglas de negocio de la fase 1 siguen validando (compatibilidad exige un vehículo).
- **Foto**: la CNN identifica la pieza. Si es de una familia del catálogo, la familia queda en la conversación y el siguiente
  mensaje de texto busca con ella; si no se vende, lo dice; si la confianza es baja, pregunta antes de seguir.

In [ ]:
from sklearn.calibration import CalibratedClassifierCV

if hasattr(modelo_texto, "predict_proba"):
    modelo_asistente = modelo_texto
else:   # el SVM lineal no da probabilidades: se calibran con validación cruzada sobre el entrenamiento
    modelo_asistente = CalibratedClassifierCV(clone(EXPERIMENTOS[MEJOR]), cv=5, method="sigmoid").fit(P_ent, y_ent)
EXACTITUD_ASISTENTE = accuracy_score(y_pru, modelo_asistente.predict(P_pru))
print(f"Modelo de texto del asistente: {type(modelo_asistente).__name__} · exactitud en la prueba {EXACTITUD_ASISTENTE:.3f}")
UMBRAL_TEXTO = 0.30
UMBRAL_IMAGEN = 0.60
REQUIERE = {i["nombre"]: set(i["requiere"]) for i in base["intenciones"]}


class AsistenteMultimodal(Chatbot):
    def __init__(self, base, catalogo, modelo, prepro, cnn):
        self.modelo, self.pre, self.cnn = modelo, prepro, cnn
        super().__init__(base, catalogo)

    def reiniciar(self):
        super().reiniciar()
        self.familia_en_curso = None

    def presentes(self, ent):
        return {k for k, v in (("referencia", ent["refs"]), ("familia", ent["familia"]),
                               ("generico", ent["generico"]), ("vehiculo", ent["vehiculo"])) if v}

    def intencion_texto(self, texto, ent):
        probs = self.modelo.predict_proba([self.pre.procesar(texto)])[0]
        orden = np.argsort(-probs)
        clases = self.modelo.classes_
        if probs[orden[0]] < UMBRAL_TEXTO:
            return "fuera_de_alcance", float(probs[orden[0]]), "confianza baja"
        for j in orden:
            if probs[j] < UMBRAL_TEXTO:
                break
            if REQUIERE.get(clases[j], set()) <= self.presentes(ent):
                return clases[j], float(probs[j]), ""
        return "fuera_de_alcance", float(probs[orden[0]]), "faltan datos"

    def responder_texto(self, texto):
        corregido = self.pre.corregir_texto(texto)
        ent = self.clasificador.entidades(corregido)
        intencion, confianza, nota = self.intencion_texto(texto, ent)
        if ent["familia"]:
            self.familia_en_curso = ent["familia"]
        elif not ent["generico"] and not ent["refs"] and self.familia_en_curso:
            if intencion == "buscar_pieza" or (intencion in ("precio", "compatibilidad") and not self.ultima_ref):
                ent["familia"] = self.familia_en_curso
        return intencion, self.responder_intencion(intencion, ent, corregido), {"confianza": confianza, "nota": nota}

    def responder_intencion(self, intencion, ent, texto):
        if intencion == "cotizar":
            return self.r_cotizar(ent, texto)
        if hasattr(self, "r_" + intencion):
            return getattr(self, "r_" + intencion)(ent)
        if intencion == "catalogo":
            conteo = Counter(p["familia"] for p in self.catalogo.values())
            return "Manejo refacciones de motor, inyección, encendido y dirección: " + \
                ", ".join(f"{self.nombres[c]} ({n})" for c, n in conteo.items()) + "."
        return self.textos[intencion].format(familias=", ".join(self.nombres.values()))

    def responder_imagen(self, ruta):
        x = keras.applications.resnet50.preprocess_input(cargar(ruta)[None, ...])
        probs = np.asarray(self.cnn.predict(x, verbose=0))[0]
        a, b = np.argsort(-probs)[:2]
        nombre, familia = CLASES[NOMBRES[a]]
        detalle = {"clase": NOMBRES[a], "confianza": float(probs[a])}
        if probs[a] < UMBRAL_IMAGEN:
            return ("confirmar_foto", f"No estoy seguro de qué pieza es: podría ser {nombre} ({probs[a]:.0%}) o "
                    f"{CLASES[NOMBRES[b]][0]} ({probs[b]:.0%}). ¿Me dices cuál es, o me mandas otra foto con mejor luz?", detalle)
        if familia:
            self.familia_en_curso = familia
            n = sum(1 for p in self.catalogo.values() if p["familia"] == familia)
            return ("foto_en_catalogo", f"En la foto veo {nombre} ({probs[a]:.0%}). Manejo {n} referencias de "
                    f"{self.nombres[familia]}. ¿Para qué vehículo y año la necesitas?", detalle)
        self.familia_en_curso = None
        return ("foto_fuera_de_catalogo", f"En la foto veo {nombre} ({probs[a]:.0%}). Esa pieza no está en nuestro catálogo; "
                f"manejo {', '.join(self.nombres.values())}. Si quieres, un asesor puede ayudarte.", detalle)

    def responder(self, texto=None, imagen=None):
        return self.responder_imagen(imagen) if imagen is not None else self.responder_texto(texto)


asistente = AsistenteMultimodal(base, catalogo, modelo_asistente, pre, cnn)
print("Asistente multimodal listo.")

In [ ]:
def foto_de(clase):
    """La primera foto de prueba de esa clase que el modelo final clasifica bien (o la primera, si ninguna)."""
    fotos = [k for k, (_, c) in enumerate(prueba) if c == clase]
    bien = [k for k in fotos if pred_img[k] == INDICE[clase]]
    return prueba[(bien or fotos)[0]][0]


k_duda = int(np.argmin(prob_final.max(axis=1)))
DEMOS = [
    ("Foto de una pieza del catálogo", [("foto", foto_de("IGNITION COIL")), "la necesito para un jetta 2005",
                                         "¿cuánto cuesta?", "cotízame 2", "gracias"]),
    ("Foto de una pieza que no se vende", [("foto", foto_de("ALTERNATOR")), "entonces quiero hablar con un asesor"]),
    ("La foto de prueba con menor confianza", [("foto", prueba[k_duda][0]), f"es {CLASES[prueba[k_duda][1]][0]}"]),
    ("Texto con faltas de ortografía", ["bomva de gazolina para tzuru 2005", "cuanto questa?", "gracias"]),
]
for titulo, turnos in DEMOS:
    asistente.reiniciar()
    print(f"\n=== {titulo} ===")
    for t in turnos:
        if isinstance(t, tuple):
            intencion, respuesta, d = asistente.responder(imagen=t[1])
            plt.figure(figsize=(2, 2))
            plt.imshow(Image.open(t[1]).convert("RGB"))
            plt.axis("off")
            plt.show()
            print(f"Usuario: [foto {os.path.basename(t[1])}]")
            print(f"  → imagen: {d['clase']} ({d['confianza']:.0%}) · {intencion}")
        else:
            intencion, respuesta, d = asistente.responder(texto=t)
            print(f"Usuario: {t}")
            print(f"  → texto: {intencion} ({d['confianza']:.0%}{', ' + d['nota'] if d['nota'] else ''})")
        print("  Bot: " + respuesta.replace("\n", "\n       "))

## 14. Las diez conversaciones del Avance 1 con el asistente integrado
Las mismas pruebas funcionales de la fase 1 (intención esperada y dato que debe traer la respuesta), ahora con el
clasificador en lugar de las reglas.

In [ ]:
fallas_int = []
for c in PRUEBAS["funcionales"]:
    asistente.reiniciar()
    for t in c["turnos"]:
        intencion, respuesta, d = asistente.responder(texto=t["usuario"])
        if not (intencion == t["intencion"] and t.get("contiene", "") in respuesta):
            fallas_int.append((c["id"], t["usuario"], t["intencion"], intencion, d["confianza"]))
TURNOS_OK = len(TURNOS) - len(fallas_int)
print(f"{len(PRUEBAS['funcionales'])} conversaciones, {len(TURNOS)} turnos · correctos: {TURNOS_OK} · fallas: {len(fallas_int)}")
for cid, u, e, o, conf in fallas_int:
    print(f"  conv. {cid} «{u}»: se esperaba {e}, respondió {o} ({conf:.0%})")

## 15. Resumen

In [ ]:
print(f"Texto · {len(textos)} frases, {len(INTENCIONES)} intenciones · modelo {MEJOR}")
print(f"  prueba: exactitud {EXACTITUD_TEXTO:.3f} · F1 macro {f1:.3f}")
print(f"  43 turnos: reglas {r_turnos}, clasificador {m_turnos} · límite: reglas {r_lim}/{len(LIMITE)}, clasificador {m_lim}/{len(LIMITE)}")
print(f"Imagen · {len(unicas)} fotos de {len(NOMBRES)} clases · ResNet50, etapa {ETAPA_FINAL}")
print(f"  prueba: exactitud {EXACTITUD_IMG:.3f} · F1 macro {fi:.3f} (A: {EXACTITUD_A:.3f}, B: {EXACTITUD_B:.3f})")
print(f"Integración · 43 turnos de la fase 1 con el asistente: {TURNOS_OK} correctos")